In [1]:
import sys
sys.path.insert(0,'..')
from warnings import filterwarnings
filterwarnings("ignore")
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
%autoreload
import os
import random
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from apex import amp
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from source.version10.data import trainLoader
from source.version10.model import ResNestModel
from source.version10.train import trainModel
from source.version10.loss import BCELoss
from catalyst.data.sampler import BalanceClassSampler

In [3]:
SEED = 42

def seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    return None

seed(42)

In [4]:
def train(fold):
    loader = {}
    loader['image_path'] = '../../data/cdeotte/train/train/'
    loader['label_path'] = '../../data/cdeotte/data.csv'
    loader['fold_idx'] = fold
    train, valid = trainLoader(**loader)
    params = {}
    params['batch_size'] = 12
    params['num_workers'] = 4
    params['drop_last'] = True
    train = DataLoader(train, shuffle=True, **params)
    valid = DataLoader(valid, **params)
    model = ResNestModel()
    model = model.to('cuda:0')
    optimizer = AdamW(model.parameters(), lr=1e-05, weight_decay=0.)
    schedular = ReduceLROnPlateau(optimizer, factor=0.5, patience=0, min_lr=1e-8)
    model, optimizer = amp.initialize(model, optimizer, opt_level='O2', verbosity=False)
    trainer = {}
    trainer['model'] = model
    trainer['train_data'] = train
    trainer['valid_data'] = valid
    trainer['loss_fn'] = BCELoss()
    trainer['optimizer'] = optimizer
    trainer['save_path'] = '../../model/version10/model_{}.pt'.format(fold)
    trainer['epochs'] = 15
    trainer['batch'] = 12
    trainer['scheduler'] = schedular
    trainModel(**trainer)
    model.cpu()
    del model
    return None

In [ ]:
train(0)

Train Images: 35419 Valid Images: 6527


Using cache found in /root/.cache/torch/hub/zhanghang1989_ResNeSt_master
 58% 20616/35412 [13:18<09:44, 25.34it/s, trn_ls=0.48580]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

100% 35412/35412 [24:13<00:00, 24.37it/s, trn_ls=0.3025, val_ls=0.1554, val_mt=0.8817]
100% 35412/35412 [24:12<00:00, 24.38it/s, trn_ls=0.2582, val_ls=0.1623, val_mt=0.9009]
  3% 1056/35412 [00:41<22:29, 25.45it/s, trn_ls=0.23610]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.

In [ ]:
train(1)

Train Images: 35397 Valid Images: 6535


Using cache found in /root/.cache/torch/hub/zhanghang1989_ResNeSt_master
 51% 17988/35388 [11:33<11:03, 26.24it/s, trn_ls=0.49650]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

 60% 21324/35388 [13:45<08:46, 26.71it/s, trn_ls=0.31420]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_msg_rate_limit`.

Current values:
NotebookApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
NotebookApp.rate_limit_window=3.0 (secs)

 68% 24132/35388 [15:33<07:04, 26.49it/s, trn_ls=0.26240]IOPub message rate exceeded.
The notebook server will temporarily stop sending output
to the client in ord

In [ ]:
train(2)

In [ ]:
train(3)

In [ ]:
train(4)